# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Modeling Method Selection:
For this task (**Lane 2: Content Refresh Opportunity Scoring**), the target variable `is_declining_label` is binary (`1` representing visibility decline, `0` otherwise). However, our ultimate objective is not just classification (yes/no), but **ranking**—we want to answer the question: *"Which pages should we refresh first?"*

Therefore, we select a **Random Forest Classifier**:
1. **Probability Output:** It generates well-calibrated class probabilities (`predict_proba`) which can be used directly as a continuous priority score to rank pages.
2. **Robustness to Non-Linearity:** Search features like GSC impressions and click rates have highly non-linear, heavy-tailed relationships with decay. Random Forests handle these shapes natively without complex scaling.
3. **Beats Simple Heuristics:** Unlike our Week-4 rule-based baseline, a Random Forest learns complex feature interactions (such as the joint impact of position, age, and scroll depth) without manual thresholding.

In [3]:
# Code cell 2: Import libraries and load data
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print(f"Dataset loaded. Total rows: {len(df)}")


Dataset loaded. Total rows: 30000


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### The Honest Split Strategy: Grouped by Client
We implement a **Grouped Client Split** using `GroupShuffleSplit` (grouped by `client_id`), holding out 25% of all clients for validation.

* **Why this is honest:** Pages belonging to the same client share common traits: site architecture, technical SEO health, domain authority, and tracking setup. If we used a standard random split, the model would memorize these client-specific baselines (memorization leakage) and fake performance. Holding out entire clients forces the model to generalize to completely unseen sites, reflecting true production deployment behavior.

In [5]:
# Code cell 4: Preprocess features and execute the Grouped Split
features = [
    "impressions_90d", "clicks_90d", "sessions_90d", "avg_position",
    "ctr", "engagement_rate", "scroll_rate", "content_age_days",
    "days_since_last_update", "word_count"
]

# Handle missing word_count by introducing a missing indicator and filling NaNs
df["word_count_missing"] = df["word_count"].isna().astype(int)
df["word_count"] = df["word_count"].fillna(df["word_count"].median())
features.append("word_count_missing")

X = df[features]
y = df["is_declining_label"]
groups = df["client_id"]

# Split grouping by client
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Training split: {X_train.shape[0]} rows (Client count: {df.iloc[train_idx]['client_id'].nunique()})")
print(f"Testing split:  {X_test.shape[0]} rows (Client count: {df.iloc[test_idx]['client_id'].nunique()})")


Training split: 22885 rows (Client count: 24)
Testing split:  7115 rows (Client count: 8)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

We train a Random Forest classifier on the training fold, predict probabilities on the testing fold, and compare its performance (Precision@50 and ROC-AUC) directly against our Week-4 baseline rule on the exact same test split.

In [7]:
# Code cell 6: Train Random Forest and generate comparison table
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Predictions
df_test = df.iloc[test_idx].copy()
df_test["model_prob"] = model.predict_proba(X_test)[:, 1]

# Calculate Week-4 baseline score on the SAME test split
def percentile_rank(series):
    return series.rank(pct=True)

df_test["visibility_score"] = percentile_rank(np.log1p(df_test["impressions_90d"]))
df_test["staleness_score"] = percentile_rank(df_test["days_since_last_update"])
df_test["baseline_score"] = 0.6 * df_test["visibility_score"] + 0.4 * df_test["staleness_score"]

# Precision@50 Evaluation
df_model_sorted = df_test.sort_values("model_prob", ascending=False)
df_baseline_sorted = df_test.sort_values("baseline_score", ascending=False)

p50_model = df_model_sorted.head(50)["is_declining_label"].mean()
p50_baseline = df_baseline_sorted.head(50)["is_declining_label"].mean()
base_rate = y_test.mean()
auc_model = roc_auc_score(y_test, df_test["model_prob"])

# Print comparison
print("==========================================================")
print("             CAPSTONE MODEL COMPARISON TABLE              ")
print("==========================================================")
print(f" Metric                  | Baseline Rule | Capstone Model ")
print("-------------------------|---------------|----------------")
print(f" Precision@50            |  {p50_baseline:.4f}       |  {p50_model:.4f}        ")
print(f" Base Rate (Random Pick) |  {base_rate:.4f}       |  {base_rate:.4f}        ")
print(f" Model ROC-AUC           |  N/A          |  {auc_model:.4f}        ")
print("==========================================================")


             CAPSTONE MODEL COMPARISON TABLE              
 Metric                  | Baseline Rule | Capstone Model 
-------------------------|---------------|----------------
 Precision@50            |  0.3200       |  0.5600        
 Base Rate (Random Pick) |  0.5165       |  0.5165        
 Model ROC-AUC           |  N/A          |  0.6083        


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Feature Importance Analysis:
The model leans heavily on the following signals to predict organic decline:
1. **`impressions_90d` (23.7% weight):** Pages with large search volume are naturally more volatile and have more search real estate to lose.
2. **`avg_position` (21.0% weight):** Position represents placement opportunity. Page 2 pages are highly unstable and prone to slipping further.
3. **`content_age_days` (16.5% weight):** The lifecycle position of the post indicates whether it is past its freshness peak.

### Error Analysis & Hard Cases (False Positives):
We inspect the top 3 false positives where the model predicted a high probability of decline, but the actual label was stable/up:
- **Case 1 (`content_3164f3076003`, prob=88.7%):** 2,696 impressions, 104 days stale, avg position 16.1.
- **Case 2 (`content_4d9f36001f06`, prob=88.0%):** 3,369 impressions, 104 days stale, avg position 13.2.
- **Case 3 (`content_41baf0722ad9`, prob=87.9%):** 3,115 impressions, 104 days stale, avg position 12.8.

* **Why these cases are hard:** All three represent mid-volume Page 2 articles that have been stale for exactly 104 days. In general, Page 2 articles that are stale are in active decay, which is why the model assigned them a high decline probability. However, these specific pages remained stable, likely because they represent highly specific niche queries with zero competition or have strong brand-loyal evergreen traffic.

In [9]:
# Code cell 8: Output feature importances and top false positives
importances = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("--- Model Feature Importances ---")
print(importances.to_string())

df_test["is_fp"] = ((df_test["is_declining_label"] == 0) & (df_test["model_prob"] >= 0.5)).astype(int)
df_fps = df_test[df_test["is_fp"] == 1].sort_values("model_prob", ascending=False)
cols_to_print = ["content_id", "model_prob", "impressions_90d", "days_since_last_update", "avg_position", "is_declining_label"]
print("\n--- Top 3 False Positives ---")
print(df_fps[cols_to_print].head(3).to_string())


--- Model Feature Importances ---
impressions_90d           0.236569
avg_position              0.209818
content_age_days          0.165291
word_count                0.075871
scroll_rate               0.062007
ctr                       0.053529
clicks_90d                0.052325
sessions_90d              0.045122
days_since_last_update    0.044455
word_count_missing        0.031450
engagement_rate           0.023560

--- Top 3 False Positives ---
                 content_id  model_prob  impressions_90d  days_since_last_update  avg_position  is_declining_label
5477   content_3164f3076003    0.886771             2696                     104          16.1                   0
12332  content_4d9f36001f06    0.880248             3369                     104          13.2                   0
20736  content_41baf0722ad9    0.879437             3115                     104          12.8                   0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.